# 01 ? Chronos-2 com LMC-Synth (seis canais)

## Registro completo do piloto

- Pergunta/escopo: verificar gera??o LMC-Synth, contrato de batches Chronos-2 e um forward/backward CPU de integra??o; n?o estimar qualidade nem declarar treino conclu?do.
- Gerador: `experiments_local/TimePFN/synthetic_data_generation/LMC_Synth.py`; `num_channels=6`, `LENGTH=1024`, `dirichlet_min=0.1`, `dirichlet_max=2.0`, `weibull_shape=2.0`, `weibul_scale=2.0`, `max_kernels=5`. Seed raiz `20261112`; seed por exemplo derivada do ID.
- Batch/modelo: 8 exemplos ? 6 canais = 48 linhas; contexto 512; horizonte 64; patch de sa?da 16; 4 output patches. Loss Chronos nativa; `future_covariates=NaN` para que as s?ries sejam alvos, n?o covari?veis futuras conhecidas.
- Checkpoint: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`; arquitetura carregada `d_model=768`, limite de contexto 8192, horizonte m?ximo 1024.
- Ambiente: Windows CPU, Python 3.11.9, PyTorch 2.12.0+cpu; CUDA indispon?vel. Fonte Chronos: checkout local `experiments_local/chronos-forecasting`.
- Resultado do smoke: batch LMC shape `[8,6,1024]`; loss finita `2.8503`, previs?o `[48,21,64]`, gradientes em 170 par?metros; backward ?10.1 s. Sem optimizer step, pesos n?o atualizados, nenhum checkpoint salvo, nenhuma m?trica de valida??o/teste.
- Comando: `python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/01_chronos2_lmc_synth.ipynb`.
- Limite/avan?o: baseline cient?fica requer treino com or?amento fixo e GPU, seguido pelo bra?o tsfmxai nos mesmos updates e protocolo do notebook 02.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/01_chronos2_lmc_synth.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.
The Chronos Python implementation is read from `experiments_local/chronos-forecasting`; recreate that ignored checkout at commit `10afa9ebe016e514f9d7dc1aa873f66af57e116b` and install it with `python -m pip install -e experiments_local/chronos-forecasting`.
LMC-Synth comes from `experiments_local/TimePFN`; recreate that ignored checkout from `https://github.com/egetaga/TimePFN.git` at commit `e11d5c2982b0e49b87873cbddcb7712641861268`.


In [1]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / "src/tsfmxai").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src/tsfmxai").exists(): raise RuntimeError('repository root not found')
sys.path.insert(0, str(ROOT / "src"))

SEED = 20261112
torch.manual_seed(SEED)
NUM_CHANNELS = 6
LENGTH = 1024
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64
EXAMPLES_PER_FORWARD = 8
MODEL_ID = "amazon/chronos-2"
MODEL_REVISION = "29ec3766d36d6f73f0696f85560a422f50e8498c"
OUTPUT_PATCH_SIZE = 16

lmc_path = ROOT / "experiments_local/TimePFN/synthetic_data_generation/LMC_Synth.py"
spec = importlib.util.spec_from_file_location("timepfn_lmc_synth", lmc_path)
lmc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lmc)
lmc.LENGTH = LENGTH
# Upstream GP draws use default_rng(None), which ignores np.random.seed.
_original_lmc_gp_sampler = lmc.sample_from_gp_prior_efficient
def _seeded_lmc_gp_sampler(kernel, X, random_seed=None, method='eigh'):
    if random_seed is None:
        random_seed = int(np.random.randint(0, 2**31 - 1))
    return _original_lmc_gp_sampler(kernel, X, random_seed=random_seed, method=method)
lmc.sample_from_gp_prior_efficient = _seeded_lmc_gp_sampler

print({"torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "seed": SEED, "channels": NUM_CHANNELS,
       "examples_per_forward": EXAMPLES_PER_FORWARD,
       "series_rows_per_forward": NUM_CHANNELS * EXAMPLES_PER_FORWARD,
       "context": CONTEXT_LENGTH, "horizon": PREDICTION_LENGTH,
       "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
       "output_patch_size": OUTPUT_PATCH_SIZE})

{'torch': '2.12.0+cpu', 'cuda_available': False, 'seed': 20261112, 'channels': 6, 'examples_per_forward': 8, 'series_rows_per_forward': 48, 'context': 512, 'horizon': 64, 'model_id': 'amazon/chronos-2', 'model_revision': '29ec3766d36d6f73f0696f85560a422f50e8498c', 'output_patch_size': 16}


C:\Users\mateu\repos\tsfmxai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the exact checkpoint once to verify its architecture/config on this machine.
chronos_src = ROOT / 'experiments_local/chronos-forecasting/src'
sys.path.insert(0, str(chronos_src))
from chronos.chronos2.pipeline import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, revision=MODEL_REVISION, device_map='cpu')
model = pipeline.model
assert model.chronos_config.output_patch_size == OUTPUT_PATCH_SIZE
assert CONTEXT_LENGTH <= model.chronos_config.context_length
assert PREDICTION_LENGTH <= model.chronos_config.max_output_patches * OUTPUT_PATCH_SIZE
print({'model_dim': model.model_dim, 'context_limit': model.chronos_config.context_length,
       'max_prediction_length': model.chronos_config.max_output_patches * OUTPUT_PATCH_SIZE,
       'output_patch_size': model.chronos_config.output_patch_size, 'loaded_device': 'cpu'})

{'model_dim': 768, 'context_limit': 8192, 'max_prediction_length': 1024, 'output_patch_size': 16, 'loaded_device': 'cpu'}


## Gera??o determin?stica de um exemplo LMC

O c?digo original sorteia usando NumPy global. Para workers reprodut?veis, isolamos o estado RNG por exemplo e restauramos o estado do processo depois da chamada. Assim, o ID do exemplo, e n?o a ordem de execu??o ou worker, define sua aleatoriedade.


In [3]:
def generate_lmc_example(example_id, *, seed=SEED):
    state = np.random.get_state()
    try:
        local_seed = int(np.random.SeedSequence([seed, example_id]).generate_state(1)[0])
        np.random.seed(local_seed)
        result = lmc.generate_time_series(
            dirichlet_min=0.1, dirichlet_max=2.0,
            weibull_shape=2.0, weibul_scale=2.0,
            max_kernels=5, num_channels=NUM_CHANNELS,
        )
        target = np.asarray(result["target"], dtype=np.float32)
        if target.shape != (NUM_CHANNELS, LENGTH):
            raise ValueError(f"Unexpected LMC shape {target.shape}")
        if not np.isfinite(target).all():
            raise ValueError("LMC generated non-finite values")
        return target
    finally:
        np.random.set_state(state)

first = generate_lmc_example(0)
assert np.array_equal(first, generate_lmc_example(0)), 'example ID must reproduce exactly'
lmc_examples = np.stack([first] + [generate_lmc_example(i) for i in range(1, EXAMPLES_PER_FORWARD)])
print("[examples, channels, time] =", lmc_examples.shape)
assert lmc_examples.shape == (EXAMPLES_PER_FORWARD, NUM_CHANNELS, LENGTH)

[examples, channels, time] = (8, 6, 1024)


## Adapter para o contrato de treino Chronos-2

O Chronos-2 espera uma linha temporal por canal e `group_ids` para indicar canais que pertencem ? mesma tarefa multivariada. O batch de 8 exemplos e 6 canais representa 48 s?ries no forward.

In [4]:
from tsfmxai.datasets.chronos2_adapter import adapt_grouped_batch
lmc_context = lmc_examples[:, :, :CONTEXT_LENGTH].transpose(0, 2, 1)
lmc_future = lmc_examples[:, :, CONTEXT_LENGTH:CONTEXT_LENGTH + PREDICTION_LENGTH].transpose(0, 2, 1)
lmc_batch = adapt_grouped_batch(
    lmc_context, lmc_future,
    num_examples=EXAMPLES_PER_FORWARD,
    channels_per_example=NUM_CHANNELS,
)
lmc_batch['num_output_patches'] = int(np.ceil(PREDICTION_LENGTH / OUTPUT_PATCH_SIZE))
assert lmc_batch['context'].shape == (48, CONTEXT_LENGTH)
assert lmc_batch['future_target'].shape == (48, PREDICTION_LENGTH)
assert lmc_batch['group_ids'].tolist() == [i for i in range(8) for _ in range(6)]
assert torch.isnan(lmc_batch['future_covariates']).all()
assert lmc_batch['num_output_patches'] == 4
print({key: (tuple(value.shape) if isinstance(value, torch.Tensor) else value)
       for key, value in lmc_batch.items() if key in ('context','future_target','future_covariates','group_ids','num_output_patches')})

{'context': (48, 512), 'future_target': (48, 64), 'future_covariates': (48, 64), 'group_ids': (48,), 'num_output_patches': 4}


In [5]:
# CPU integration smoke only: one LMC forward/backward, no optimizer update or checkpoint.
model.train()
model.zero_grad(set_to_none=True)
t0=__import__('time').perf_counter()
output=model(context=lmc_batch['context'],group_ids=lmc_batch['group_ids'],
             future_covariates=lmc_batch['future_covariates'],
             num_output_patches=lmc_batch['num_output_patches'],
             future_target=lmc_batch['future_target'])
assert output.loss is not None and torch.isfinite(output.loss)
assert output.quantile_preds.shape==(48,21,64)
output.loss.backward()
gradient_tensors=sum(parameter.grad is not None for parameter in model.parameters())
assert gradient_tensors>0
print({'cpu_lmc_forward_backward_s':round(__import__('time').perf_counter()-t0,3),
       'loss':float(output.loss.detach()),'prediction_shape':tuple(output.quantile_preds.shape),
       'parameters_with_gradients':gradient_tensors,'optimizer_update':False})
model.zero_grad(set_to_none=True)
model.eval(); print('model returned to eval mode')

{'cpu_lmc_forward_backward_s': 10.883, 'loss': 2.8502824306488037, 'prediction_shape': (48, 21, 64), 'parameters_with_gradients': 170, 'optimizer_update': False}
model returned to eval mode


## Resultado desta etapa e gates para treino

A execu??o gera e adapta batches LMC-Synth com IDs determin?sticos, verifica o agrupamento dos 48 canais e carrega o checkpoint Chronos-2 fixado. O treino completo segue pendente: requer GPU, n?mero de updates congelado, splits fixos e execu??o do mesmo or?amento no bra?o tsfmxai. `future_covariates=NaN` marca todos os canais como alvos a prever; zeros seriam tratados como covari?veis conhecidas e mascarariam a loss.

O smoke test de CPU valida apenas a conex?o de dados, loss e gradientes num batch. Ele n?o ajusta pesos, n?o seleciona checkpoints e n?o ? resultado da baseline.